In [ ]:
import os
import pickle
import pandas as pd
from openpyxl.styles.builtins import total
from sympy.physics.units import charge
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import json
import seaborn as sns
from scipy.interpolate import PchipInterpolator
from pathlib import Path
from scipy.interpolate import splrep, splev
from scipy import interpolate
import matplotlib.pyplot as plt

from sympy.physics.control.control_plots import matplotlib

# BatteryLife datasets
# CALB, CALCE, HNEI, HUST, ISU_ILCC, MATR, MICH, MICH_EXP, NA-ion, RWTH, SDU, SNL, Stanford_2, Tongji, UL_PUR, XJTU, ZN-coin

path = '/data/trf/python_works/BatteryLife/dataset/ISU_ILCC/'
ppath = Path(path)
files = os.listdir(path)
files = [i for i in files if i.endswith('.pkl')]
soc = []
cell = []
life_label = []
file_name = []
total_soh = []
total_cycles = []
data = {}
for file in tqdm(files):
    print(file)
    file_name.append(file)
    with open(path + f'{file}', 'rb') as f:
        cell_data = pickle.load(f)
        filename = file.split('.pkl')[0]
        length = len(cell_data['cycle_data'])
        cell = cell_data['cycle_data']
        nominal_capacity = cell_data['nominal_capacity_in_Ah']


        df = pd.DataFrame()
        SOC_interval = cell_data['SOC_interval']  # get the charge and discharge soc interval

        SOC_interval = SOC_interval[1] - SOC_interval[0]


        times = []
        cycles = []
        for i in range(0, length):
            cycle_df = pd.DataFrame()
            cycle_data_len = len(cell_data['cycle_data'][i])
            cycle_data = cell_data['cycle_data'][i]
            cycle_df['current'] = cycle_data['current_in_A']
            cycle_df['voltage'] = cycle_data['voltage_in_V']
            cycle_df['charge_capacity'] = cycle_data['charge_capacity_in_Ah']
            cycle_df['discharge_capacity'] = cycle_data['discharge_capacity_in_Ah']
            cycle_df['test_time_s'] = cycle_data['time_in_s']
            cycle_df['cycle_number'] = cycle_data['cycle_number']
            cycles.append(i+1)

            if file.startswith('CALB_-10'):
                soh_value = abs(cycle_df.loc[cycle_df['current'] < 0, 'discharge_capacity'].min())
            elif 'DefaultGroup' in file:
                soh_value = float(cycle_df['discharge_capacity'].max())
            else:
                soh_value = float(cycle_df.loc[cycle_df['current'] < 0, 'discharge_capacity'].max())


            if SOC_interval == 0:
                SOC_interval = 1

            time = cycle_df['test_time_s'].tolist()
            time = [round(i, 6) for i in time]
            diff = np.diff(time)

            # find is there any abnormal data in time columns
            bad_idx = np.where(diff < 0)[0]

            if len(bad_idx) != 0:
                print(f"time of cycle {i+1} in cell {filename}, The following positions are not in ascending order.：")
                for i in bad_idx:
                    print(f"arr[{i}]={time[i]} -> arr[{i + 1}]={time[i + 1]}")
            times += time

            times += cycle_df['test_time_s'].tolist()


        # fig = plt.figure(figsize=(10, 4))
        # plt.xlabel('Cycle number')
        # plt.ylabel('SOH')
        # plt.grid(alpha=.3)
        # plt.plot(range(len(times)), times)
        # plt.title(f'{filename}')
        # plt.show()



  0%|          | 0/240 [00:00<?, ?it/s]

ISU-ILCC_G23C3.pkl
time of cycle 510 in cell ISU-ILCC_G23C3, The following positions are not in ascending order.：
arr[21]=5475038.0 -> arr[22]=5475036.0


  0%|          | 1/240 [00:01<07:50,  1.97s/it]

ISU-ILCC_G64C4.pkl
time of cycle 1094 in cell ISU-ILCC_G64C4, The following positions are not in ascending order.：
arr[259]=6282095.0 -> arr[260]=6282094.0
time of cycle 1247 in cell ISU-ILCC_G64C4, The following positions are not in ascending order.：
arr[633]=6843112.0 -> arr[634]=6843106.0
time of cycle 2128 in cell ISU-ILCC_G64C4, The following positions are not in ascending order.：
arr[172]=8745558.0 -> arr[173]=8745549.0


  1%|          | 2/240 [00:09<21:50,  5.51s/it]

...

ISU-ILCC_G43C4.pkl
time of cycle 96 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[491]=265288.0 -> arr[492]=265287.0
time of cycle 507 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[542]=1371138.0 -> arr[543]=1371129.0
time of cycle 1665 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[253]=4333576.0 -> arr[254]=4333550.0
time of cycle 2580 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[121]=6461340.0 -> arr[122]=6461338.0
time of cycle 3063 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[149]=7489884.0 -> arr[150]=7489882.0
time of cycle 3340 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[342]=8040935.0 -> arr[343]=8040933.0
time of cycle 5927 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[197]=11797017.0 -> arr[198]=11796974.0
time of cycle 7606 in cel

 25%|██▍       | 59/240 [08:42<34:35, 11.47s/it]

time of cycle 14249 in cell ISU-ILCC_G43C4, The following positions are not in ascending order.：
arr[58]=15871270.0 -> arr[59]=15871237.0
ISU-ILCC_G64C1.pkl


 25%|██▌       | 60/240 [08:46<27:01,  9.01s/it]

time of cycle 1310 in cell ISU-ILCC_G64C1, The following positions are not in ascending order.：
arr[56]=5179649.0 -> arr[57]=5179648.0
ISU-ILCC_G54C1.pkl
time of cycle 295 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[302]=863943.0 -> arr[303]=863936.0
time of cycle 1293 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[178]=3776006.0 -> arr[179]=3775974.0
time of cycle 1585 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[164]=4615811.0 -> arr[165]=4615806.0
time of cycle 1906 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[518]=5539297.0 -> arr[519]=5539289.0
time of cycle 2785 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[251]=7972641.0 -> arr[252]=7972637.0
time of cycle 3467 in cell ISU-ILCC_G54C1, The following positions are not in ascending order.：
arr[324]=9671298.0 -> arr[325]=9671295.0
time of cycle 5645 in cell 